# RetailRocket EDA

Dataset: [RetailRocket E-Commerce](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset)  
Files: `events.csv`, `item_properties_part1/2.csv`, `category_tree.csv`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

%matplotlib inline
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (10, 4)})

RAW = '../data/raw'

## 1. Events

In [ ]:
events = pd.read_csv(f'{RAW}/events.csv')
events['timestamp'] = pd.to_datetime(events['timestamp'], unit='ms')
print(events.shape)
events.head()

In [ ]:
print(events.dtypes)
print('\nNulls:')
print(events.isnull().sum())

In [ ]:
etype = events['event'].value_counts()
print(etype)
print(f'\nview→addtocart rate: {etype["addtocart"]/etype["view"]:.2%}')
print(f'addtocart→transaction rate: {etype["transaction"]/etype["addtocart"]:.2%}')

fig, ax = plt.subplots()
etype.plot(kind='bar', ax=ax, color=['steelblue','orange','green'])
ax.set_title('Event type distribution')
ax.set_ylabel('Count')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
print(f'Period: {events["timestamp"].min()}  →  {events["timestamp"].max()}')
print(f'Duration: {(events["timestamp"].max() - events["timestamp"].min()).days} days')

daily = events.set_index('timestamp').resample('D').size()
daily.plot(title='Daily events', ylabel='Events')
plt.tight_layout()

In [ ]:
user_counts = events.groupby('visitorid').size()
print(f'Unique users  : {len(user_counts):,}')
print(f'Interactions/user (median): {user_counts.median():.0f}')
print(f'Interactions/user (mean)  : {user_counts.mean():.1f}')
print(f'Max interactions (single user): {user_counts.max():,}')

fig, ax = plt.subplots()
ax.hist(user_counts.clip(upper=50), bins=50, edgecolor='white')
ax.set_title('Interactions per user (clipped at 50)')
ax.set_xlabel('Interactions')
ax.set_ylabel('Users')
plt.tight_layout()

In [ ]:
item_counts = events.groupby('itemid').size()
print(f'Unique items  : {len(item_counts):,}')
print(f'Interactions/item (median): {item_counts.median():.0f}')
print(f'Top-10 items:')
print(item_counts.sort_values(ascending=False).head(10))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
item_counts.clip(upper=100).hist(bins=50, ax=axes[0], edgecolor='white')
axes[0].set_title('Interactions per item (clipped at 100)')
axes[0].set_xlabel('Interactions')

# Log-log power-law check
sorted_counts = item_counts.sort_values(ascending=False).reset_index(drop=True)
axes[1].loglog(sorted_counts.values)
axes[1].set_title('Item popularity (log-log)')
axes[1].set_xlabel('Item rank')
axes[1].set_ylabel('Interactions')
plt.tight_layout()

## 2. Sparsity

In [ ]:
n_users = events['visitorid'].nunique()
n_items = events['itemid'].nunique()
n_interactions = len(events)
sparsity = 1 - n_interactions / (n_users * n_items)

print(f'Users       : {n_users:,}')
print(f'Items       : {n_items:,}')
print(f'Interactions: {n_interactions:,}')
print(f'Matrix size : {n_users * n_items:,}')
print(f'Sparsity    : {sparsity:.4%}')

# Users with >=5 interactions (filter threshold)
active_users = (user_counts >= 5).sum()
print(f'\nUsers with >=5 interactions: {active_users:,} ({active_users/n_users:.1%})')

## 3. Item Properties

In [ ]:
# Load sample from each part to avoid OOM (files are ~450MB + ~390MB)
props1 = pd.read_csv(f'{RAW}/item_properties_part1.csv', nrows=500_000)
props2 = pd.read_csv(f'{RAW}/item_properties_part2.csv', nrows=500_000)
props = pd.concat([props1, props2], ignore_index=True)
print(props.shape)
props.head()

In [ ]:
print('Unique items in sample:', props['itemid'].nunique())
print('Unique properties     :', props['property'].nunique())

top_props = props['property'].value_counts().head(20)
print('\nTop 20 properties:')
print(top_props)

fig, ax = plt.subplots(figsize=(10, 5))
top_props.plot(kind='barh', ax=ax)
ax.set_title('Top 20 item properties')
ax.invert_yaxis()
plt.tight_layout()

In [ ]:
# Category property coverage
has_category = props[props['property'] == 'categoryid']['itemid'].nunique()
total_items_in_props = props['itemid'].nunique()
print(f'Items with categoryid: {has_category:,} / {total_items_in_props:,} ({has_category/total_items_in_props:.1%})')

# Timestamp range in properties
props['timestamp'] = pd.to_datetime(props['timestamp'], unit='ms')
print(f'Properties period: {props["timestamp"].min()}  →  {props["timestamp"].max()}')

## 4. Category Tree

In [ ]:
cats = pd.read_csv(f'{RAW}/category_tree.csv')
print(cats.shape)
cats.head(10)

In [ ]:
print(f'Total categories: {cats["categoryid"].nunique():,}')
print(f'Root categories (no parent): {cats["parentid"].isnull().sum():,}')

# Depth analysis via BFS
parent_map = dict(zip(cats['categoryid'], cats['parentid']))
def depth(cat_id: int, max_depth: int = 20) -> int:
    d = 0
    while parent_map.get(cat_id) and d < max_depth:
        cat_id = parent_map[cat_id]
        d += 1
    return d

cats['depth'] = cats['categoryid'].apply(depth)
print('\nDepth distribution:')
print(cats['depth'].value_counts().sort_index())